# **EDA Notebook**



---
## 0. Setup Environment

In [1]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
group_name = "Group 52"
student_name = "Andres Ethorimn Lobarb"
student_id = "26029165"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [6]:
# No additional packages: pandas, numpy, scipy and altair are available on Colab.

### 0.b Import Packages

In [7]:
import pandas as pd
import altair as alt
import numpy as np
from scipy.stats import kruskal, norm, spearmanr
from scipy.optimize import curve_fit
from IPython.display import display

In [8]:
# Utils

def column_profile(df):
    """One row per column: type, missing count and share, distinct values."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_share": df.isna().mean().round(4),
        "distinct": df.nunique(),
    })


def past_and_future_change(balances, account_col, date_col, balance_col, horizon):
    """Change over the `horizon` month-ends before and after each month-end, per account."""
    ordered = balances.sort_values([account_col, date_col])
    grouped = ordered.groupby(account_col)[balance_col]
    out = ordered.copy()
    out["past_change"] = ordered[balance_col] - grouped.shift(horizon)
    out["future_change"] = grouped.shift(-horizon) - ordered[balance_col]
    return out


def spearman_summary(x, y, label):
    """Spearman correlation with a Fisher-z 95% interval and a size label."""
    mask = x.notna() & y.notna()
    rho, p_value = spearmanr(x[mask], y[mask])
    n = int(mask.sum())
    half_width = 1.96 / np.sqrt(n - 3)
    low, high = np.tanh(np.arctanh(rho) - half_width), np.tanh(np.arctanh(rho) + half_width)
    size = abs(rho)
    magnitude = "negligible" if size < 0.1 else "small" if size < 0.3 else "medium" if size < 0.5 else "large"
    return pd.Series({"feature": label, "n": n, "rho": round(rho, 4), "ci_low": round(low, 4),
                      "ci_high": round(high, 4), "p_value": p_value, "magnitude": magnitude})


def holm_adjust(p_values):
    """Holm step-down adjustment of a list of p-values (family-wise error control)."""
    order = np.argsort(p_values)
    m = len(p_values)
    adjusted = np.empty(m)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p_values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted


def kruskal_epsilon(df, value_col, group_col):
    """Kruskal-Wallis H test across groups with epsilon-squared effect size."""
    groups = [g[value_col].dropna().to_numpy() for _, g in df.groupby(group_col)]
    h_stat, p_value = kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    epsilon_sq = (h_stat - k + 1) / (n - k)
    return pd.Series({"H": round(h_stat, 2), "groups": k, "n": n, "p_value": p_value,
                      "epsilon_squared": round(epsilon_sq, 6)})


def wilson_interval(successes, n):
    """95% Wilson interval for a proportion."""
    z = 1.96
    p = successes / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


def qq_frame(values):
    """Sample quantiles against normal quantiles (all points up to 500 values, else 199 levels).

    Robust standardisation (median, IQR / 1.349) keeps a few extreme values from flattening the plot;
    for normal data it equals the usual standardisation."""
    values = pd.Series(values).dropna()
    n = len(values)
    levels = (np.arange(1, n + 1) - 0.5) / n if n <= 500 else np.linspace(0.005, 0.995, 199)
    scale = (values.quantile(0.75) - values.quantile(0.25)) / 1.349
    scale = scale if scale > 0 else values.std()  # IQR is 0 when most values are equal
    z = (values - values.median()) / scale
    return pd.DataFrame({"theoretical": norm.ppf(levels), "sample": np.quantile(z, levels)})


def distribution_charts(values, label, n_bins, tail):
    """Histogram (the `tail` share at each end not drawn) beside a normal QQ plot, with shape statistics."""
    values = pd.Series(values).dropna()
    low, high = values.quantile(tail), values.quantile(1 - tail)
    shown = values[(values >= low) & (values <= high)]
    counts, edges = np.histogram(shown, bins=n_bins)
    hist = alt.Chart(pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})).mark_bar(
        color="#2a78d6").encode(x=alt.X("start:Q", bin="binned", title=label), x2="end:Q",
                                y=alt.Y("count:Q", title="Count")).properties(width=300, height=220)
    diagonal = pd.DataFrame({"theoretical": [-3, 3], "sample": [-3, 3]})
    qq = (alt.Chart(qq_frame(values)).mark_point(color="#2a78d6", size=14).encode(
            x=alt.X("theoretical:Q", title="Normal quantile"), y=alt.Y("sample:Q", title="Robust-standardised sample quantile"))
          + alt.Chart(diagonal).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="theoretical:Q", y="sample:Q")
          ).properties(width=300, height=220)
    print(f"{label}: n = {len(values)}, skewness = {values.skew():.2f}, excess kurtosis = {values.kurt():.2f}, "
          f"histogram omits {len(values) - len(shown)} values outside the {tail:.1%} to {1 - tail:.1%} quantiles")
    return alt.hconcat(hist, qq)

In [9]:
# Constants
# <HORIZON> is a provisional working horizon (months ahead) used where one horizon is needed;
# B.2 examines the horizons in LAGS before any horizon is chosen.
HORIZON = 3
LAGS = [1, 3, 6]
DROP_LEVELS = [0.05, 0.10, 0.20]   # relative drops examined in B.4; none is chosen here
FIRST_CUTOFF = pd.Timestamp("2023-03-31")   # first month-end with 3 months of transactions before it (SB-1 B.1)
BALANCE_EDGES = 10 ** np.arange(0, 7, 0.25)  # log-spaced balance bins for B.6, from $1 to $10 million

---
## B. Data Understanding

**Approach.** We first check the quality of the balance history (B.1), then answer the questions the forecast design depends on: how far ahead a forecast is informative (B.2), whether account types behave differently (B.3), how often large drops happen and what signals them (B.4), what each horizon leaves us to work with (B.5), and how drop behaviour depends on the balance (B.6). Each finding comes from the outputs above it; we take the decisions after the EDA notebooks.

In [10]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "account_balance_history.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [13]:
df["balance_date"] = pd.to_datetime(df["balance_date"])
print("Rows, columns:", df.shape)
print("Accounts:", df["account_id"].nunique(), "| month-ends:", df["balance_date"].nunique(),
      "| from", df["balance_date"].min().date(), "to", df["balance_date"].max().date())
print("Month-ends per account:", df.groupby("account_id").size().value_counts().to_dict())
print("Duplicated (account_id, balance_date):", df.duplicated(["account_id", "balance_date"]).sum())

Rows, columns: (1850976, 8)
Accounts: 38562 | month-ends: 48 | from 2022-01-31 to 2025-12-31
Month-ends per account: {48: 38562}
Duplicated (account_id, balance_date): 0


In [14]:
display(column_profile(df))

,type,missing,missing_share,distinct
history_id,int64,0,0.0,1850976
account_id,int64,0,0.0,38562
customer_id,int64,0,0.0,15000
balance_date,datetime64[ns],0,0.0,48
balance,float64,0,0.0,1819919
balance_band,object,0,0.0,5
balance_change,float64,0,0.0,1090087
stress_indicator,int64,0,0.0,16


In [17]:
ordered = df.sort_values(["account_id", "balance_date"])
diff = ordered.groupby("account_id")["balance"].diff()
consistent = np.isclose(diff, ordered["balance_change"], atol=0.01)[diff.notna()]
print("balance_change equals the month-on-month difference (within $0.01):", round(consistent.mean(), 4),
      "of", int(diff.notna().sum()), "checkable rows")

display(df["balance"].describe().round(2).to_frame("balance ($)"))
print("Zero balances:", (df["balance"] == 0).sum(), "| negative balances:", (df["balance"] < 0).sum())
print("Balance range by balance_band ($):")
display(df.groupby("balance_band")["balance"].agg(["min", "max", "size"]).sort_values("min").round(0))
print("stress_indicator values (share of rows):")
display(df["stress_indicator"].value_counts(normalize=True).sort_index().round(4).to_frame("share").T)

balance_change equals the month-on-month difference (within $0.01): 0.9986 of 1812414 checkable rows


,balance ($)
count,1850976.00
mean,405200.73
std,496172.09
min,0.00
25%,76899.46
50%,208402.37
75%,457776.98
max,3652885.50


Zero balances: 2584 | negative balances: 0
Balance range by balance_band ($):


,min,max,size
balance_band,,,
Low,0.0,1000.0,4170
Medium,1002.0,10000.0,25893
High,10000.0,50000.0,254722
Premium,50000.0,250000.0,759178
Private Banking,250000.0,3652886.0,807013


stress_indicator values (share of rows):


stress_indicator,0,10,15,20,25,35,40,45,50,60,65,70,75,85,90,95
share,0.5056,0.2753,0.0171,0.1376,0.033,0.0207,0.0008,0.0077,0.0011,0.0006,0.0001,0.0002,0.0,0.0001,0.0,0.0


In [19]:
# Complementary load: account attributes to check history against the account's life
accounts = pd.read_csv(at.folder_path / "accounts.csv", usecols=["account_id", "account_type", "open_date",
                                                                  "account_status", "current_balance"])
accounts["open_date"] = pd.to_datetime(accounts["open_date"])
with_accounts = df.merge(accounts, on="account_id", how="left")
before_open = with_accounts["balance_date"] < with_accounts["open_date"]
print("Month-ends dated before the account's open_date:", int(before_open.sum()), f"({before_open.mean():.1%})")
last = with_accounts[with_accounts["balance_date"] == with_accounts["balance_date"].max()]
print("Last month-end balance equals accounts.current_balance (within $1):",
      round(np.isclose(last["balance"], last["current_balance"], atol=1).mean(), 4))
print("Spread of the monthly change by balance quintile ($, standard deviation):")
display(ordered.groupby(pd.qcut(ordered["balance"], 5))["balance_change"].std().round(0).to_frame("std of monthly change"))

Month-ends dated before the account's open_date: 278116 (15.0%)
Last month-end balance equals accounts.current_balance (within $1): 0.0001
Spread of the monthly change by balance quintile ($, standard deviation):


,std of monthly change
balance,
"(-0.001, 62079.58]",2327.0
"(62079.58, 148068.78]",3228.0
"(148068.78, 274950.98]",4101.0
"(274950.98, 682790.11]",5425.0
"(682790.11, 3652885.5]",7715.0


In [20]:
monthly = (df.groupby("balance_date")["balance"].agg(mean="mean", median="median").reset_index()
           .melt("balance_date", var_name="statistic", value_name="balance"))
alt.Chart(monthly).mark_line(point=True).encode(
    x=alt.X("balance_date:T", title="Month-end"),
    y=alt.Y("balance:Q", title="Balance across accounts ($)", scale=alt.Scale(zero=False)),
    color=alt.Color("statistic:N", title=None, scale=alt.Scale(range=["#2a78d6", "#898781"])),
    tooltip=["balance_date:T", "statistic", alt.Tooltip("balance:Q", format=".0f")],
).properties(width=620, height=240)

alt.Chart(...)

In [22]:
dataset_insights = """
The history is complete: 1850976 rows, every one of the 38562 accounts with all 48 month-ends from
2022-01-31 to 2025-12-31, and no missing values or duplicates. Balances range from $0 to $3.65
million (median $208402, mean $405201), 85% of the rows sit in Premium or Private Banking, and the
average balance rises slowly from about $400000 to $410000 (chart). <stress_indicator> turns out
to be an undefined score (16 values from 0 to 95) rather than a flag.

Findings:

1. 15.0% of month-ends predate the account's <open_date>, so they are not real history.
2. <balance_change> disagrees with the balance difference in 0.14% of rows; we recompute it.
3. <accounts.current_balance> matches the last balance for only 0.01% of accounts.
4. Larger balances move more in dollars ($2327 to $7715 by quintile), so we also read dollar
   errors by balance segment.
"""

In [23]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `future_change` at 1, 3 and 6 months

In [24]:
# Change from each month-end to the month-end 1, 3 and 6 months later, per account
changes_by_lag = {lag: past_and_future_change(df, "account_id", "balance_date", "balance", lag) for lag in LAGS}
alt.vconcat(*[distribution_charts(changes_by_lag[lag]["future_change"], f"Balance change over {lag} month(s) ($)", 40, 0.005)
              for lag in LAGS])

Balance change over 1 month(s) ($): n = 1812414, skewness = 0.10, excess kurtosis = 3.43, histogram omits 18126 values outside the 0.5% to 99.5% quantiles
Balance change over 3 month(s) ($): n = 1735290, skewness = 0.16, excess kurtosis = 3.38, histogram omits 17354 values outside the 0.5% to 99.5% quantiles
Balance change over 6 month(s) ($): n = 1619604, skewness = 0.24, excess kurtosis = 3.36, histogram omits 16198 values outside the 0.5% to 99.5% quantiles


alt.VConcatChart(...)

In [25]:
# How much does "the balance stays the same" miss at each horizon, and does the recent movement
# say anything about the next one?
rows, tests = [], []
for lag in LAGS:
    frame = changes_by_lag[lag]
    future = frame["future_change"].dropna()
    rows.append({"horizon_months": lag, "examples": len(future),
                 "persistence_mae": round(future.abs().mean(), 1),
                 "mean_change": round(future.mean(), 1),
                 "mae_if_mean_change_added": round((future - future.mean()).abs().mean(), 1)})
    tests.append(spearman_summary(frame["past_change"], frame["future_change"], f"past vs next {lag} months"))
horizon_table = pd.DataFrame(rows)
test_table = pd.DataFrame(tests).set_index("feature")
test_table["p_holm"] = holm_adjust(test_table["p_value"].to_numpy())
display(horizon_table)
display(test_table)

monthly_change = ordered.assign(prev=ordered.groupby("account_id")["balance_change"].shift(1))
print("Correlation of one month's change with the previous month's change:",
      round(monthly_change["balance_change"].corr(monthly_change["prev"]), 4))

,horizon_months,examples,persistence_mae,mean_change,mae_if_mean_change_added
0,1,1812414,3464.6,232.9,3462.5
1,3,1735290,6018.7,699.9,6007.7
2,6,1619604,8547.1,1401.3,8515.9


,n,rho,ci_low,ci_high,p_value,magnitude,p_holm
feature,,,,,,,
past vs next 1 months,1773852,0.0008,-0.0007,0.0023,2.787822e-01,negligible,2.787822e-01
past vs next 3 months,1619604,0.0030,0.0015,0.0046,1.128554e-04,negligible,2.257108e-04
past vs next 6 months,1388232,0.0076,0.0060,0.0093,2.621645e-19,negligible,7.864936e-19


Correlation of one month's change with the previous month's change: 0.0006


In [26]:
mae_chart = alt.Chart(horizon_table).mark_bar(color="#2a78d6").encode(
    x=alt.X("horizon_months:O", title="Months ahead"),
    y=alt.Y("persistence_mae:Q", title="Average miss of 'balance stays the same' ($)"),
    tooltip=["horizon_months", "persistence_mae", "mae_if_mean_change_added"],
).properties(width=260, height=240)
rho_table = test_table.reset_index().assign(horizon_months=LAGS)
rho_chart = (alt.Chart(rho_table).mark_rule(color="#898781").encode(
        x=alt.X("horizon_months:O", title="Months ahead"),
        y=alt.Y("ci_low:Q", title="Spearman rho, past vs next change", scale=alt.Scale(domain=[-0.1, 0.1])),
        y2="ci_high:Q")
    + alt.Chart(rho_table).mark_point(filled=True, color="#2a78d6").encode(
        x="horizon_months:O", y="rho:Q", tooltip=["feature", "rho", "ci_low", "ci_high"])
    + alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#898781", strokeDash=[4, 4]).encode(y="y:Q")
    + alt.Chart(pd.DataFrame({"y": [0], "label": ["no relation"]})).mark_text(
        align="left", dx=4, dy=-7, color="#898781").encode(y="y:Q", x=alt.value(0), text="label:N")
).properties(width=260, height=240)
alt.hconcat(mae_chart, rho_chart)

alt.HConcatChart(...)

In [27]:
feature_1_insights = """
Assuming that "the balance stays the same" misses by $3465 on average one month ahead, $6019 at
three months and $8547 at six (left chart): the error grows with the square root of the horizon.
Changes are symmetric, with heavier tails than a normal curve (plots above).

What an account did recently tells us almost nothing about what it does next: the correlation
between past and next change stays negligible at every horizon (Spearman rho = 0.0008, 0.0030 and
0.0076, Holm-corrected; right chart).
"""

In [28]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `account_type`

In [29]:
by_type = (past_and_future_change(df, "account_id", "balance_date", "balance", HORIZON)
           .merge(accounts[["account_id", "account_type"]], on="account_id").dropna(subset=["future_change"]))

# Examples per account type, and the shape of the change within each type
type_counts = by_type["account_type"].value_counts().rename_axis("account_type").reset_index(name="examples")
bars = alt.Chart(type_counts).mark_bar(color="#2a78d6").encode(
    x=alt.X("examples:Q", title="Examples"), y=alt.Y("account_type:N", title="Account type")).properties(width=300, height=220)
qq = pd.concat([qq_frame(group["future_change"]).assign(account_type=kind) for kind, group in by_type.groupby("account_type")])
diagonal = pd.DataFrame({"theoretical": [-3, 3], "sample": [-3, 3]})
qq_by_type = (alt.Chart(qq).mark_line().encode(
        x=alt.X("theoretical:Q", title="Normal quantile"),
        y=alt.Y("sample:Q", title=f"Robust-standardised {HORIZON}-month change quantile"),
        color=alt.Color("account_type:N", title="Account type"))
    + alt.Chart(diagonal).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="theoretical:Q", y="sample:Q")
).properties(width=300, height=220)
print(by_type.groupby("account_type")["future_change"].agg(skewness="skew", excess_kurtosis=lambda s: s.kurt()).round(2))
alt.hconcat(bars, qq_by_type)

              skewness  excess_kurtosis
account_type                           
Business          0.15             3.75
Offset            0.18             3.42
Savings           0.16             3.34
Term Deposit      0.14             3.23
Transaction       0.15             3.42
Youth             0.24             3.31


alt.HConcatChart(...)

In [31]:
# Do account types move differently over the working horizon?
summary = by_type.groupby("account_type").agg(
    accounts=("account_id", "nunique"), examples=("future_change", "size"), mean=("future_change", "mean"),
    std=("future_change", "std"), median=("future_change", "median"),
    zero_change_share=("future_change", lambda s: (s == 0).mean())).round(4)
display(summary)
type_test = kruskal_epsilon(by_type, "future_change", "account_type")
display(type_test.to_frame("Kruskal-Wallis, change over the next HORIZON months by account type"))

,accounts,examples,mean,std,median,zero_change_share
account_type,,,,,,
Business,1901,85545,653.6591,8586.7696,413.710,0.0004
Offset,3796,170820,656.8908,8557.0070,430.570,0.0003
Savings,13650,614250,715.8895,8537.5999,470.655,0.0005
Term Deposit,5846,263070,706.8980,8536.7883,468.155,0.0004
Transaction,11502,517590,696.3860,8570.6486,459.495,0.0004
Youth,1867,84015,716.9819,8556.2479,443.980,0.0005


,"Kruskal-Wallis, change over the next HORIZON months by account type"
H,1.197000e+01
groups,6.000000e+00
n,1.735290e+06
p_value,3.513787e-02
epsilon_squared,4.000000e-06


In [32]:
quantiles = (by_type.groupby("account_type")["future_change"]
             .quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack().reset_index())
quantiles.columns = ["account_type", "p05", "p25", "p50", "p75", "p95"]
base = alt.Chart(quantiles).encode(y=alt.Y("account_type:N", title="Account type"))
(base.mark_rule(color="#898781").encode(x=alt.X("p05:Q", title=f"Balance change over the next {HORIZON} months ($), 5th to 95th percentile"), x2="p95:Q")
 + base.mark_bar(color="#2a78d6", height=14).encode(x="p25:Q", x2="p75:Q")
 + base.mark_tick(color="white", thickness=2, size=14).encode(x="p50:Q")
).properties(width=560, height=220)

alt.LayerChart(...)

In [34]:
feature_2_insights = """
All six account types move the same way: their QQ curves overlap, the average 3-month change runs
from $654 to $717, the spread from $8537 to $8587, and months without any change stay at or below
0.05%, Term Deposits included (chart).

The difference between types is negligible (Kruskal-Wallis epsilon squared = 0.000004; the p of
0.035 only reflects the 1735290 examples). In this data the account type does not separate how
balances behave.
"""

In [35]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `relative_change` (large drops)

In [ ]:
drops = past_and_future_change(df, "account_id", "balance_date", "balance", HORIZON).dropna(subset=["future_change"])
drops = drops[drops["balance"] > 0].copy()
drops["relative_change"] = drops["future_change"] / drops["balance"]

distribution_charts(drops["relative_change"], f"Relative balance change over {HORIZON} months", 40, 0.005)

In [ ]:
# How often does a balance fall sharply over the working horizon, and does <stress_indicator> signal it?
level_rows = []
for level in DROP_LEVELS:
    hits = int((drops["relative_change"] <= -level).sum())
    low, high = wilson_interval(hits, len(drops))
    level_rows.append({"drop_of_at_least": f"{level:.0%}", "examples": hits, "share": round(hits / len(drops), 4),
                       "ci_low": round(low, 4), "ci_high": round(high, 4)})
display(pd.DataFrame(level_rows))

band_rows = []
for band, group in drops.groupby("balance_band"):
    hits = int((group["relative_change"] <= -0.10).sum())
    low, high = wilson_interval(hits, len(group))
    band_rows.append({"balance_band": band, "examples": len(group), "median_balance": round(group["balance"].median(), 0),
                      "share_drop_10pct": round(hits / len(group), 4), "ci_low": round(low, 4), "ci_high": round(high, 4),
                      "mean_stress": round(group["stress_indicator"].mean(), 2)})
band_table = pd.DataFrame(band_rows).sort_values("median_balance")
display(band_table)
stress_test = spearman_summary(drops["stress_indicator"], drops["relative_change"], "stress_indicator vs relative change")
display(stress_test.to_frame("Spearman, stress_indicator vs relative change over the next HORIZON months"))

In [ ]:
overall_share = float((drops["relative_change"] <= -0.10).mean())
ref = pd.DataFrame({"x": [overall_share], "label": [f"all month-ends: {overall_share:.2%}"]})
base = alt.Chart(band_table).encode(y=alt.Y("balance_band:N", title="Balance band", sort=list(band_table["balance_band"])))
(base.mark_rule(color="#898781").encode(x=alt.X("ci_low:Q", title=f"Share of month-ends followed by a drop of 10% or more within {HORIZON} months"), x2="ci_high:Q")
 + base.mark_point(filled=True, color="#e34948").encode(x="share_drop_10pct:Q",
       tooltip=["balance_band", "examples", "share_drop_10pct", "ci_low", "ci_high", "mean_stress"])
 + alt.Chart(ref).mark_rule(color="#898781", strokeDash=[4, 4]).encode(x="x:Q")
 + alt.Chart(ref).mark_text(align="left", dx=4, color="#898781").encode(x="x:Q", y=alt.value(0), text="label:N")
).properties(width=520, height=200)

In [ ]:
feature_n_insights = """
A drop of 10% or more within 3 months follows 4.25% of month-ends (10.2% for 5%, 1.45% for 20%).
These relative changes are extremely skewed by near-zero balances (plots above).

The drops concentrate in small balances: they follow 37.2% of month-ends in the Low band and 39.7%
in Medium, against 17.5% in High, 3.0% in Premium and 0.05% in Private Banking (chart).
<stress_indicator> is higher in the Low band (mean 32.75), yet it does not warn of a drop
(Spearman rho = 0.001, 95% interval -0.0004 to 0.0025).
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

### B.5 Explore Feature of Interest `horizon` (what each horizon leaves to work with)

In [ ]:
# Per horizon: usable cut-offs (from the first month-end with 3 months of transactions), examples,
# the miss of "balance stays the same", the typical relative movement and the share of 10% drops.
rows = []
for lag in LAGS:
    frame = changes_by_lag[lag].dropna(subset=["future_change"])
    usable = frame[(frame["balance_date"] >= FIRST_CUTOFF) & (frame["balance"] > 0)]
    relative = usable["future_change"] / usable["balance"]
    rows.append({"horizon_months": lag, "usable_cutoffs": usable["balance_date"].nunique(), "examples": len(usable),
                 "persistence_mae": round(usable["future_change"].abs().mean(), 1),
                 "median_abs_relative_change": round(relative.abs().median(), 4),
                 "share_drop_10pct": round((relative <= -0.10).mean(), 4)})
tradeoff = pd.DataFrame(rows)
display(tradeoff)

In [ ]:
cutoff_chart = alt.Chart(tradeoff).mark_bar(color="#2a78d6").encode(
    x=alt.X("horizon_months:O", title="Months ahead"), y=alt.Y("usable_cutoffs:Q", title="Usable monthly cut-offs"),
    tooltip=["horizon_months", "usable_cutoffs", "examples"]).properties(width=260, height=220)
drop_chart = alt.Chart(tradeoff).mark_bar(color="#e34948").encode(
    x=alt.X("horizon_months:O", title="Months ahead"),
    y=alt.Y("share_drop_10pct:Q", title="Share of month-ends followed by a 10% drop", axis=alt.Axis(format="%")),
    tooltip=["horizon_months", "share_drop_10pct", "median_abs_relative_change"]).properties(width=260, height=220)
alt.hconcat(cutoff_chart, drop_chart)

In [ ]:
feature_5_insights = """
Each horizon leaves a different amount to work with: 33 usable cut-offs at one month, 31 at three
and 28 at six. Drops of 10% or more follow 1.93%, 4.22% and 6.50% of month-ends, and balances
typically move by 1.1%, 1.9% and 2.8% (table, chart).

A longer horizon gives the bank more drops to act on, but fewer periods to learn and test on and a
larger miss of "balance stays the same" ($3469, $6020 and $8548).
"""

In [ ]:
print_tile(size="h3", key='feature_5_insights', value=feature_5_insights)

### B.6 Explore Feature of Interest `balance` (does drop behaviour change below some balance?)

In [ ]:
# Bins on a log scale of the cut-off balance (drops frame from B.4: positive balances only).
# A natural floor would show as a break in how the dollar spread of the change scales with the balance.
levels = drops.assign(bin=pd.cut(drops["balance"], BALANCE_EDGES, right=False))
by_level = levels.groupby("bin", observed=True).agg(
    median_balance=("balance", "median"), examples=("balance", "size"),
    change_std=("future_change", "std"),
    share_drop_10pct=("relative_change", lambda s: (s <= -0.10).mean()),
    relative_iqr=("relative_change", lambda s: s.quantile(0.75) - s.quantile(0.25))).reset_index(drop=True)
by_level = by_level[by_level["examples"] >= 30]   # bins with too few examples give unstable spreads
slope, intercept = np.polyfit(np.log10(by_level["median_balance"]), np.log10(by_level["change_std"]), 1)
by_level["std_vs_fit"] = by_level["change_std"] / 10 ** (intercept + slope * np.log10(by_level["median_balance"]))
display(by_level.round(4))
print(f"Dollar spread of the {HORIZON}-month change scales as balance^{slope:.3f}; "
      f"relative spread therefore scales as balance^{slope - 1:.3f}")

# Where does the drop share fall? Logistic curve on log10 balance, weighted by examples:
# share = plateau / (1 + exp(rate * (log10 balance - midpoint))). The midpoint is the balance where
# the share is half the plateau; it comes from the data, not from a chosen value.
def logistic(x, plateau, rate, midpoint):
    return plateau / (1 + np.exp(rate * (x - midpoint)))

x = np.log10(by_level["median_balance"].to_numpy())
params, _ = curve_fit(logistic, x, by_level["share_drop_10pct"].to_numpy(), p0=[0.4, 3, 4.3],
                      sigma=1 / np.sqrt(by_level["examples"].to_numpy()))
plateau, rate, midpoint = params
bank_rate = float((drops["relative_change"] <= -0.10).mean())
at_bank_rate = midpoint + np.log(plateau / bank_rate - 1) / rate
print(f"Plateau {plateau:.1%}; share halves at a balance of ${10 ** midpoint:.0f}; "
      f"reaches the overall rate ({bank_rate:.2%}) at ${10 ** at_bank_rate:.0f}")

In [ ]:
fit_line = pd.DataFrame({"median_balance": by_level["median_balance"],
                         "fitted": 10 ** (intercept + slope * np.log10(by_level["median_balance"]))})
spread = (alt.Chart(by_level).mark_point(filled=True, color="#2a78d6").encode(
        x=alt.X("median_balance:Q", title="Balance at cut-off ($, log scale)", scale=alt.Scale(type="log")),
        y=alt.Y("change_std:Q", title=f"Spread of the {HORIZON}-month change ($, log scale)", scale=alt.Scale(type="log")),
        tooltip=["median_balance", "examples", "change_std", "std_vs_fit"])
    + alt.Chart(fit_line).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="median_balance:Q", y="fitted:Q")
).properties(width=300, height=240)
overall_share = float((drops["relative_change"] <= -0.10).mean())
share = (alt.Chart(by_level).mark_point(filled=True, color="#e34948").encode(
        x=alt.X("median_balance:Q", title="Balance at cut-off ($, log scale)", scale=alt.Scale(type="log")),
        y=alt.Y("share_drop_10pct:Q", title=f"Share followed by a 10% drop within {HORIZON} months", axis=alt.Axis(format="%")),
        tooltip=["median_balance", "examples", "share_drop_10pct", "relative_iqr"])
    + alt.Chart(pd.DataFrame({"median_balance": 10 ** np.linspace(x.min(), x.max(), 100)}).assign(
        fitted=lambda d: logistic(np.log10(d["median_balance"]), *params))).mark_line(color="#898781").encode(
        x="median_balance:Q", y="fitted:Q")
    + alt.Chart(pd.DataFrame({"x": [10 ** midpoint]})).mark_rule(color="#898781", strokeDash=[4, 4]).encode(x="x:Q")
    + alt.Chart(pd.DataFrame({"y": [overall_share]})).mark_rule(color="#898781", strokeDash=[2, 2]).encode(y="y:Q")
).properties(width=300, height=240)
alt.hconcat(spread, share)

In [ ]:
feature_6_insights = """
Below a balance of about $2500, accounts move by roughly the same dollar amount (a spread of about
$2150 to $2700); above it, the spread grows far more slowly than the balance (slope 0.195 on log
scales), so relative movements shrink as balances grow (left chart).

The share of month-ends followed by a 10% drop traces an S-curve (right chart): a plateau of 42.5%
for small balances, half of that at $24005, and the overall 4.25% at $84870. Bins below $30 hold
fewer than 30 examples and are left out.

For small balances a 10% drop is routine rather than a warning, and the change happens gradually
between about $8000 and $85000 instead of at one obvious floor.
"""

In [ ]:
print_tile(size="h3", key='feature_6_insights', value=feature_6_insights)

## C. Ethical Consideration of Features


In [ ]:
# Who sits in the lowest balance bands? A first look at whether low balances concentrate
# in particular customer groups (complementary load of customers).
customers = pd.read_csv(at.folder_path / "customers.csv", usecols=["customer_id", "age", "gender", "vulnerable_customer_flag"])
latest = df[df["balance_date"] == df["balance_date"].max()].merge(customers, on="customer_id", how="left")
display(latest.groupby("balance_band").agg(accounts=("account_id", "size"), mean_age=("age", "mean"),
                                           vulnerable_share=("vulnerable_customer_flag", "mean")).round(3))

In [ ]:
ethical_considerations = """
The Low and Medium bands hold more vulnerable customers (5.8% and 4.5%, against 3.1% to 3.4% in
the other bands), so anything aimed at small balances reaches them more often. Such a use may only
trigger support, never a restriction, a fee or a credit decision, and we audit it by customer
group before it is used.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)